# Silver Layer — Data Quality Checks

This notebook defines reusable data quality check functions used across the Silver layer transformation notebooks.

Each function validates a specific property of a transformed DataFrame before it is written to a Silver Delta table. On success the function prints a PASSED message; on failure it raises a `ValueError` with details, stopping the pipeline.

## Checks Defined Here

* `check_not_null` — verifies that specified columns have no null or blank values
* `check_unique` — verifies that the given key (single or composite) has no duplicates
* `check_row_count` — verifies that the Silver row count falls within an expected percentage range of the Bronze source

## How To Use

Import these functions into a Silver transformation notebook by running:
​```
%run ../Utilities/silver_dq_checks
​```
Then call the functions on the transformed DataFrame before writing to Silver.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### check_not_null
Validates that the specified columns contain no null or blank/whitespace values.

In [0]:
def check_not_null (df,columns):
    ''' 
    Checks if the columns in the dataframe are empty or not.
    If it is null or empty then raise ValueError else return None.
    '''
    results ={}
    for column_name in columns:
        null_count = df.filter((col(column_name).isNull()) | (trim(col(column_name)) == '')).count() # count null values for column_name
        if null_count > 0:
            results[column_name] = null_count

    if len(results) == 0:
        print('check_not_null: PASSED')
        return None
    else:
        raise ValueError(f'check_not_null FAILED: Nulls found. The following are the pairs of null value columns and its null counts - {results}')

### check_unique
Validates that the specified key columns produce unique rows. Supports single-column or composite keys.

In [0]:
def check_unique (df,columns):
    '''
    Checks if the columns in the dataframe are unique or not. It based on one key or composite keys.
    If it is not unique then raise ValueError else return None.
    '''
    duplicate_df = df.groupBy(*columns).count().filter(col('count') > 1)
    duplicate_count =duplicate_df.count()
    duplicate_keys = duplicate_df.drop('count').take(5)

    if duplicate_count == 0:
        print('check_unique: PASSED')
        return None
    else:
        raise ValueError(f'check_unique FAILED: Duplicates found.The duplicates found on {columns}, total keys duplicated are {duplicate_count} and the some of them are listed here: {duplicate_keys}')

### check_row_count
Validates that the Silver row count is within an acceptable percentage range of the Bronze source row count.


In [0]:
def check_row_count(df,bronze_table,min_pct,max_pct):
    '''
    Checks if the silver table has the numbers of rows between our expected range.
    '''
    bronze_count = spark.read.table(bronze_table).count()

    if bronze_count == 0:
        raise ValueError(f"check_row_count FAILED: Bronze table {bronze_table} has 0 rows — cannot compute retention.")

    else:
        silver_count = df.count()
        if (silver_count >= (bronze_count * (min_pct/100))) and (silver_count <= (bronze_count * (max_pct/100))):
            print('check_row_count: PASSED')
            return None
        else:
            actual_pct = (silver_count/bronze_count)*100
            raise ValueError(f'check_row_count FAILED. Bronze count: {bronze_count}, Silver count: {silver_count}, Actual%: {actual_pct:.2f} % and expected range between:  {min_pct} and {max_pct}')


### check_event_sequence
Validates that timestamp columns appear in the expected chronological order. Skips rows where either timestamp in a pair is null. Raises on any violation.

In [0]:
def check_event_sequence(df_silver,ordered_columns):
    '''
    Checks if the events in the silver table are in the correct order.
    ordered_columns is a list of columns in the correct order
    '''

    violation_by_pairs ={}

    for i in range(len(ordered_columns)-1):
        col1 = ordered_columns[i]
        col2 = ordered_columns[i+1]

        # filter for the rows where both columns are not null and the first column is greater than the second column. which means Violation.
    
        violation_df = df_silver.filter((col(col1).isNotNull()) & (col(col2).isNotNull()) & (col(col1) > col(col2)))
        violation_count = violation_df.count()

        if violation_count > 0:
            violation_by_pairs[f"{col1}-{col2}"] = violation_count
    
    
    if len(violation_by_pairs) > 0:
        raise ValueError(f'check_event_sequence: FAILED. Found {len(violation_by_pairs)} violations in the following columns pair: {violation_by_pairs}')
    else:
        print('check_event_sequence: PASSED')
        return None

### identify_event_sequence_violations
Adds one boolean column per consecutive timestamp pair to mark whether each row violates the expected order. Returns the annotated DataFrame without raising — callers use it to split clean rows from violations.

In [0]:
def identify_event_sequence_violations(df,ordered_columns):
    '''
    Returns the original DataFrame with one additional boolean column per
    consecutive pair, named '_violates_{col1}_{col2}'. The column is True
    for rows where both columns are non-null AND col1 > col2..
    '''
    for i in range(len(ordered_columns) - 1):
        col1 = ordered_columns[i]
        col2 = ordered_columns[i + 1]
        new_col_name = f"_violates_{col1}_{col2}"
        df = df.withColumn(new_col_name, 
            col(col1).isNotNull() & 
            col(col2).isNotNull() & 
            (col(col1) > col(col2))
    )
    return df